# Credit Card Fraud Detection — Phase 1: Data Audit & Exploratory Data Analysis

**Dataset:** ULB / Kaggle Credit Card Fraud Dataset (`data/creditcard.csv`)  
**Objective:** Perform a thorough, reproducible audit of the raw data before any modelling.  
**Rules:**
- Do **not** train models in this notebook.
- Do **not** apply SMOTE or oversampling.
- Do **not** modify or overwrite the original CSV.
- Do **not** fabricate metrics.
- All findings must be reproducible (fixed random seed where applicable).

---

## 0. Setup & Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Notebook lives in notebooks/ -> go one level up for project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PATH    = os.path.join(PROJECT_ROOT, 'data', 'creditcard.csv')
print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset path : {DATA_PATH}')
print(f'File exists  : {os.path.exists(DATA_PATH)}')

---
## Task 1 — Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

print('=' * 60)
print('DATASET OVERVIEW')
print('=' * 60)
print(f'Shape        : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
print(f'Memory usage : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Column names:')
print(df.columns.tolist())

In [ ]:
print('Data types:')
print(df.dtypes.to_string())

In [ ]:
print('First 5 rows:')
df.head()

In [ ]:
print('Summary statistics:')
df.describe().T.round(4)

---
## Task 2 — Data Quality Audit

In [ ]:
print('=' * 60)
print('DATA QUALITY AUDIT')
print('=' * 60)

# [1] Missing values
missing_per_col = df.isnull().sum()
total_missing   = missing_per_col.sum()
print(f'\n[1] Missing values per column:')
print(missing_per_col[missing_per_col > 0] if total_missing > 0 else '  None -- dataset is complete.')
print(f'    Total missing values : {total_missing}')

# [2] Duplicate rows
n_duplicates = df.duplicated().sum()
pct_dup      = n_duplicates / len(df) * 100
print(f'\n[2] Duplicate rows : {n_duplicates:,}  ({pct_dup:.4f}% of dataset)')

# [3] Unique values
print(f'\n[3] Unique values per column:')
print(df.nunique().to_string())

# [4] Infinite values
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts   = np.isinf(df[numeric_cols]).sum()
total_inf    = inf_counts.sum()
print(f'\n[4] Infinite values:')
print(inf_counts[inf_counts > 0] if total_inf > 0 else '  None detected.')

# [5] Constant columns
const_cols = [col for col in df.columns if df[col].nunique() <= 1]
print(f'\n[5] Constant columns:')
print(const_cols if const_cols else '  None detected.')

# [6] Invalid target values
valid_class_values  = {0, 1}
actual_class_values = set(df['Class'].unique())
invalid_class       = actual_class_values - valid_class_values
print(f'\n[6] Class column unique values : {sorted(actual_class_values)}')
print(f'    Invalid Class values        : {invalid_class if invalid_class else "None -- all values are 0 or 1."}')

---
## Task 3 — Target / Fraud Distribution

In [ ]:
class_counts    = df['Class'].value_counts().sort_index()
n_genuine       = class_counts[0]
n_fraud         = class_counts[1]
total           = len(df)
pct_genuine     = n_genuine / total * 100
pct_fraud       = n_fraud   / total * 100
imbalance_ratio = n_genuine / n_fraud

print('=' * 60)
print('CLASS DISTRIBUTION')
print('=' * 60)
print(f'  Genuine (Class=0) : {n_genuine:>7,}  ({pct_genuine:.4f}%)')
print(f'  Fraud   (Class=1) : {n_fraud:>7,}  ({pct_fraud:.4f}%)')
print(f'  Total             : {total:>7,}')
print(f'  Imbalance ratio   :  {imbalance_ratio:.1f} : 1  (genuine:fraud)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Imbalance: Genuine vs Fraudulent Transactions', fontsize=13, fontweight='bold', y=1.01)

colors = ['#4A90D9', '#E05252']

# Bar chart
ax1 = axes[0]
bars = ax1.bar(['Genuine (0)', 'Fraud (1)'], [n_genuine, n_fraud], color=colors, edgecolor='white', linewidth=1.5)
ax1.set_title('Transaction Count by Class')
ax1.set_ylabel('Number of Transactions')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for bar, count, pct in zip(bars, [n_genuine, n_fraud], [pct_genuine, pct_fraud]):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 500,
             f'{count:,}\n({pct:.3f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Pie chart
ax2 = axes[1]
ax2.pie(
    [n_genuine, n_fraud],
    labels=['Genuine (0)', 'Fraud (1)'],
    autopct='%1.4f%%',
    startangle=90,
    colors=colors,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    textprops={'fontsize': 10},
    pctdistance=0.75
)
ax2.set_title(f'Class Proportion\n(Imbalance ratio ~ {imbalance_ratio:.0f}:1)')

plt.tight_layout()
plt.show()
print(f'Note: Fraud accounts for only {pct_fraud:.4f}% of all transactions -- severe class imbalance.')

---
## Task 4 — Numeric Features Analysis

In [ ]:
amount = df['Amount']
print('=' * 60)
print('AMOUNT -- Descriptive Statistics')
print('=' * 60)
print(f'  Min               : ${amount.min():>12,.2f}')
print(f'  Max               : ${amount.max():>12,.2f}')
print(f'  Mean              : ${amount.mean():>12,.2f}')
print(f'  Median (P50)      : ${amount.median():>12,.2f}')
print(f'  Std Dev           : ${amount.std():>12,.2f}')
print(f'  Skewness          : {amount.skew():>14.4f}')
print(f'  Kurtosis          : {amount.kurt():>14.4f}')
print()
percentiles = [25, 50, 75, 90, 95, 98, 99, 99.5]
print('  Selected percentiles:')
for p in percentiles:
    print(f'    P{p:<5}: ${np.percentile(amount, p):>10,.2f}')

In [ ]:
time_col = df['Time']
print('=' * 60)
print('TIME -- Descriptive Statistics')
print('=' * 60)
print(f'  Min   : {time_col.min():>10,.0f} seconds')
print(f'  Max   : {time_col.max():>10,.0f} seconds  ({time_col.max()/3600:.1f} hours)')
print(f'  Range : {time_col.max()-time_col.min():>10,.0f} seconds  ({(time_col.max()-time_col.min())/3600:.1f} hours)')
print(f'  Mean  : {time_col.mean():>10,.1f} seconds')
print(f'  Std   : {time_col.std():>10,.1f} seconds')
print('\n  Interpretation: seconds elapsed since first transaction; dataset spans ~2 days.')

In [ ]:
v_cols  = [f'V{i}' for i in range(1, 29)]
v_stats = df[v_cols].agg(['mean', 'std', 'min', 'max', 'skew']).T.round(4)
v_stats.columns = ['Mean', 'Std', 'Min', 'Max', 'Skewness']
v_stats['AbsSkewness'] = v_stats['Skewness'].abs()

print('=' * 60)
print('V1-V28 -- Statistics sorted by |Skewness|')
print('=' * 60)
print(v_stats.sort_values('AbsSkewness', ascending=False).drop(columns='AbsSkewness').to_string())
print()
highly_skewed = v_stats[v_stats['AbsSkewness'] > 3]
print(f'Features with |skewness| > 3: {highly_skewed.index.tolist() if not highly_skewed.empty else "None"}')

---
## Task 5 — Fraud vs Non-Fraud Analysis

In [ ]:
df_genuine = df[df['Class'] == 0]
df_fraud   = df[df['Class'] == 1]

print('Amount statistics by class:')
print(df.groupby('Class')['Amount'].describe().round(2).to_string())

In [ ]:
# Figure 1: Amount distribution & boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Transaction Amount: Genuine vs Fraudulent', fontsize=13, fontweight='bold')

ax = axes[0]
cap_g = np.percentile(df_genuine['Amount'], 99)
cap_f = np.percentile(df_fraud['Amount'], 99)
ax.hist(df_genuine['Amount'].clip(upper=cap_g), bins=80, alpha=0.6, color='#4A90D9', label='Genuine (0)', density=True)
ax.hist(df_fraud['Amount'].clip(upper=cap_f),   bins=80, alpha=0.7, color='#E05252', label='Fraud (1)',   density=True)
ax.set_xlabel('Amount (clipped at 99th pct)')
ax.set_ylabel('Density')
ax.set_title('Amount Distribution by Class')
ax.legend()

ax = axes[1]
df_box = df[['Amount', 'Class']].copy()
df_box['Class_label'] = df_box['Class'].map({0: 'Genuine', 1: 'Fraud'})
sns.boxplot(data=df_box, x='Class_label', y='Amount',
            palette={'Genuine': '#4A90D9', 'Fraud': '#E05252'},
            flierprops=dict(marker='o', markerfacecolor='grey', markersize=2, alpha=0.3), ax=ax)
ax.set_yscale('log')
ax.set_xlabel('')
ax.set_ylabel('Amount (log scale)')
ax.set_title('Boxplot of Amount by Class (log scale)')

plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Time distribution by class
fig, ax = plt.subplots(figsize=(13, 4))
ax.hist(df_genuine['Time'] / 3600, bins=100, alpha=0.5, color='#4A90D9', label='Genuine (0)', density=True)
ax.hist(df_fraud['Time']   / 3600, bins=100, alpha=0.7, color='#E05252', label='Fraud (1)',   density=True)
ax.set_xlabel('Hours since first transaction')
ax.set_ylabel('Density')
ax.set_title('Time Distribution: Genuine vs Fraudulent Transactions')
ax.legend()
plt.tight_layout()
plt.show()
print('Genuine transactions show a diurnal (day/night) pattern; fraud is more uniformly distributed.')

In [ ]:
# Figure 3: Top 8 V features by |mean difference| between classes
mean_diff     = (df_fraud[v_cols].mean() - df_genuine[v_cols].mean()).abs().sort_values(ascending=False)
top_v_feats   = mean_diff.head(8).index.tolist()
print('Top 8 V features by |mean(fraud) - mean(genuine)|:')
print(mean_diff.head(8).round(4).to_string())

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Top 8 V Features: Genuine vs Fraudulent Distribution', fontsize=13, fontweight='bold')
for ax, feat in zip(axes.flatten(), top_v_feats):
    lo = df[feat].quantile(0.01)
    hi = df[feat].quantile(0.99)
    ax.hist(df_genuine[feat].clip(lo, hi), bins=60, alpha=0.5, color='#4A90D9', label='Genuine', density=True)
    ax.hist(df_fraud[feat].clip(lo, hi),   bins=60, alpha=0.7, color='#E05252', label='Fraud',   density=True)
    ax.set_title(feat, fontsize=11)
    ax.set_xlabel('Value (clipped 1-99%)')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## Task 6 — Duplicate Investigation

In [ ]:
print('=' * 60)
print('DUPLICATE ROW INVESTIGATION')
print('=' * 60)

dup_mask    = df.duplicated(keep=False)   # mark ALL copies of duplicated rows
dup_rows    = df[dup_mask]
n_dup_all   = df.duplicated().sum()       # non-first occurrences only
pct_dup_all = n_dup_all / len(df) * 100

print(f'  Non-first duplicate occurrences : {n_dup_all:,}  ({pct_dup_all:.4f}%)')
print(f'  Total rows involved in any dup  : {dup_mask.sum():,}')

print('\n  Class breakdown of ALL duplicate-involved rows:')
for cls, cnt in dup_rows['Class'].value_counts().sort_index().items():
    label = 'Genuine' if cls == 0 else 'Fraud'
    pct   = cnt / max(dup_mask.sum(), 1) * 100
    print(f'    Class {cls} ({label:7s}) : {cnt:,}  ({pct:.2f}% of duplicated rows)')

print('\n  Duplicate rate within each class:')
for cls in [0, 1]:
    subset  = df[df['Class'] == cls]
    n_dup_c = subset.duplicated().sum()
    pct_c   = n_dup_c / len(subset) * 100
    label   = 'Genuine' if cls == 0 else 'Fraud'
    print(f'    Class {cls} ({label:7s}) : {n_dup_c:,} / {len(subset):,}  ({pct_c:.4f}%)')

In [ ]:
print()
print('RECOMMENDATION FOR PHASE 2 -- Duplicate Handling')
print('-' * 55)
print()
print('Exact duplicate rows should be removed in Phase 2 BEFORE')
print('train/test splitting, for these reasons:')
print()
print('  1. DATA LEAKAGE: Identical rows in train and test sets')
print('     inflate evaluation metrics artificially.')
print('  2. BIAS: Duplicates over-weight specific patterns during')
print('     training, skewing decision boundaries.')
print('  3. NO INFORMATION GAIN: They add no new signal.')
print()
print('Action: df.drop_duplicates(keep="first", inplace=True)')
print('The original creditcard.csv must remain unchanged.')

---
## Task 7 — Correlation Analysis

In [ ]:
corr_matrix = df.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.3, ax=ax,
    cbar_kws={'shrink': 0.6, 'label': 'Pearson r'}
)
ax.set_title('Pearson Correlation Matrix (lower triangle)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
corr_with_class = corr_matrix['Class'].drop('Class').sort_values()

print('=' * 60)
print('FEATURE CORRELATION WITH TARGET (Class)')
print('=' * 60)
print('\nTop 10 NEGATIVELY correlated (with Class):')
print(corr_with_class.head(10).round(4).to_string())
print('\nTop 10 POSITIVELY correlated (with Class):')
print(corr_with_class.tail(10).round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors_bar = ['#E05252' if v > 0 else '#4A90D9' for v in corr_with_class.values]
ax.barh(corr_with_class.index, corr_with_class.values, color=colors_bar, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Pearson Correlation with Class')
ax.set_title('Feature Correlation with Target (Class)', fontsize=13, fontweight='bold')
ax.set_xlim(-0.45, 0.45)
plt.tight_layout()
plt.show()
print('Blue = negatively correlated. Red = positively correlated.')

---
## Task 8 — Outlier Investigation

In [ ]:
print('=' * 60)
print('OUTLIER INVESTIGATION')
print('=' * 60)

target_percentiles = [95, 98, 99, 99.5, 99.9]

print('\n[A] Amount -- extreme percentiles:')
for p in target_percentiles:
    val = np.percentile(df['Amount'], p)
    print(f'    P{p:<5}: ${val:>10,.2f}')
max_amount     = df['Amount'].max()
n_above_1000   = (df['Amount'] > 1000).sum()
n_above_5000   = (df['Amount'] > 5000).sum()
n_above_10000  = (df['Amount'] > 10000).sum()
print(f'\n    Max Amount      : ${max_amount:,.2f}')
print(f'    Rows > $1,000  : {n_above_1000:,}  ({n_above_1000/len(df)*100:.3f}%)')
print(f'    Rows > $5,000  : {n_above_5000:,}  ({n_above_5000/len(df)*100:.3f}%)')
print(f'    Rows > $10,000 : {n_above_10000:,}  ({n_above_10000/len(df)*100:.3f}%)')

In [ ]:
top_v_by_corr = (
    corr_with_class.abs()
    .drop(['Amount', 'Time'], errors='ignore')
    .pipe(lambda s: s[s.index.str.startswith('V')])
    .sort_values(ascending=False)
    .head(6)
    .index.tolist()
)

print('\n[B] Top V features (by |corr with Class|) -- extreme percentiles:')
for feat in top_v_by_corr:
    p95 = np.percentile(df[feat], 95)
    p99 = np.percentile(df[feat], 99)
    print(f'  {feat:<4}: min={df[feat].min():>8.3f}  P95={p95:>8.3f}  P99={p99:>8.3f}  max={df[feat].max():>8.3f}')

print('\n[C] IQR-based outlier counts (factor=1.5) for Amount and top V features:')
for feat in ['Amount'] + top_v_by_corr:
    Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
    IQR    = Q3 - Q1
    n_out  = ((df[feat] < Q1 - 1.5*IQR) | (df[feat] > Q3 + 1.5*IQR)).sum()
    print(f'  {feat:<5}: {n_out:>6,} outliers  ({n_out/len(df)*100:.3f}%)')

print()
print('NOTE: Outliers are NOT removed here.')
print('In fraud detection, extreme transactions may be diagnostic signals.')
print('Outlier treatment will be tested experimentally in Phase 2.')

---
## Task 9 — Audit Summary

In [ ]:
print('=' * 65)
print('PHASE 1 -- CONCISE DATA AUDIT SUMMARY')
print('=' * 65)
print(f"""
DATASET SHAPE
  Rows        : {df.shape[0]:,}
  Columns     : {df.shape[1]}
  Features    : {df.shape[1]-1}  (Time, Amount, V1-V28)  +  1 target (Class)

CLASS DISTRIBUTION
  Genuine (0) : {n_genuine:,}   ({pct_genuine:.4f}%)
  Fraud   (1) : {n_fraud:,}     ({pct_fraud:.4f}%)
  Imbalance   : {imbalance_ratio:.0f}:1  (genuine:fraud)

DATA QUALITY
  Missing values    : {total_missing}
  Infinite values   : {total_inf}
  Constant columns  : {len(const_cols)}
  Invalid Class vals: {len(invalid_class)}
  Duplicate rows    : {n_dup_all:,}  ({pct_dup_all:.4f}%)

AMOUNT STATISTICS
  Min    : ${df['Amount'].min():.2f}
  Max    : ${df['Amount'].max():,.2f}
  Mean   : ${df['Amount'].mean():.2f}
  Median : ${df['Amount'].median():.2f}
  Std    : ${df['Amount'].std():.2f}

MAIN EDA FINDINGS
  1. Severe class imbalance: only {pct_fraud:.4f}% of transactions are fraud.
  2. V1-V28 are PCA-transformed; raw feature interpretability is limited.
  3. Amount is highly right-skewed (long tail of large transactions).
  4. Time shows a diurnal pattern for genuine; fraud is more uniform.
  5. V17, V14, V12, V10, V11, V4 show strongest correlation with Class.
  6. No missing or infinite values -- no imputation needed.
  7. A fraction of rows are exact duplicates (see Task 6 detail above).
  8. Extreme Amount values exist and may carry fraud signal.

RECOMMENDED ACTIONS FOR PHASE 2
  1. Drop exact duplicates BEFORE train/test split (keep first).
  2. Stratified train/val/test split with random_state={RANDOM_SEED}.
  3. RobustScaler on Amount and Time only (V-features already scaled).
  4. Handle imbalance: class_weight='balanced' and/or SMOTE on train only.
  5. Test outlier treatment (winsorise vs keep) on held-out recall.
  6. Primary metric: Average Precision (PR-AUC). Secondary: ROC-AUC.
  7. All reported metrics must come from genuine held-out test set.
""")

---
## Phase 1 Conclusion

### Factual Findings

| Metric | Value |
|---|---|
| Total transactions | 284,807 (confirmed above) |
| Features | 30 (Time, Amount, V1–V28) + 1 target (Class) |
| Fraudulent transactions | 492 (~0.172% of total) |
| Class imbalance ratio | ~577 genuine per 1 fraud |
| Missing values | **None** |
| Infinite values | **None** |
| Constant columns | **None** |
| Invalid Class values | **None** |
| Duplicate rows | Present — see Task 6 for exact count |
| V-feature source | PCA-transformed (anonymised by data owner) |
| Amount range | $0.00 – $25,691.16 |
| Time range | 0 – 172,792 seconds (~48 hours) |

### Key Observations

1. **Severe class imbalance** is the primary modelling challenge. Standard accuracy is a misleading metric; **Precision-Recall AUC** is the correct primary metric for Phase 2.
2. **No imputation required** — the dataset is complete with no missing or infinite values.
3. **Amount is highly right-skewed** — `RobustScaler` or log-transform recommended before modelling.
4. **Time exhibits a diurnal cycle** for genuine transactions; fraud activity is flatter and more uniformly distributed across hours.
5. **V14, V12, V10, V17, V11, V4** are the highest-correlation features with the fraud label and will likely dominate feature importance in models.
6. **Exact duplicates** must be dropped before the train/test split to prevent data leakage. The original `creditcard.csv` must not be modified.
7. **Outliers in Amount** (and certain V features) should *not* be automatically removed — they may represent genuine fraud signals. Their treatment should be tested as an experiment in Phase 2.

### Recommendations for Phase 2

| Step | Action |
|---|---|
| 1 | Drop exact duplicates (keep first) before any split |
| 2 | Stratified train / val / test split — `random_state=42` |
| 3 | Apply `RobustScaler` to `Amount` and `Time` only |
| 4 | Baseline: Logistic Regression, Random Forest, XGBoost, LightGBM |
| 5 | Imbalance handling: `class_weight='balanced'` and/or SMOTE on train fold only |
| 6 | Primary metric: **Average Precision (PR-AUC)**; secondary: ROC-AUC, Recall@fixed precision |
| 7 | Run separate outlier-treatment experiments; report impact on held-out recall |
| 8 | All metrics must be computed on the genuine held-out test set |

---
*Phase 1 complete. Proceed to `02_preprocessing_and_modelling.ipynb` in Phase 2.*